### **Week 11 Monday: psycopg - Integrating Python with PostgreSQL**

### Today's Objectives

- Understand the role of a database adapter/driver.

- Learn to connect to a PostgreSQL database from a Python application using `psycopg`.

- Execute SQL commands (CRUD) from Python and handle the results.

- Implement proper connection management and error handling.

- Safely pass parameters to SQL queries to prevent SQL injection.

### Prerequisites

- Knowledge from Week 9 (SQL CRUD, DB Design).

- A running PostgreSQL database (e.g., on localhost).

- The `psycopg` library installed (`pip install psycopg-binary`).

### **1. Introduction: Why do we need `psycopg`**?

### The Problem
Python doesn't natively speak the PostgreSQL protocol. We need a "translator" or a "driver".

### The Solution: DB-API & Psycopg

- **DB-API (PEP 249):** A standard specification for Python database interfaces. It defines a common API for all database connectors.
- **Psycopg:** The most popular, efficient, and feature-rich PostgreSQL adapter for Python. It implements the DB-API spec.

**Analogy:**
- **Python** is like you.
- **PostgreSQL** is a librarian who only speaks French.
- **`psycopg`** is your bilingual friend who translates your questions (Python) into French (PostgreSQL protocol) and brings back the answers.

### **2. Installation & Basic Connection**

### 2.1 Installation

```bash
pip install "psycopg[binary]"
```


### 2.2 The Basic Connection Pattern

**⚠️ Critical:** Always manage your connections properly! Use a `try...except...finally` block or a context manager (`with`).

In [ ]:
import psycopg
from psycopg import OperationalError

# Connection parameters - UPDATE THESE FOR YOUR DB!
DB_NAME = "task_manager"
DB_USER = "clement"                  # postgres
DB_PASSWORD = "mypassword"           # server connection password
DB_HOST = "localhost"                # or your server's IP
DB_PORT = "5432"

# 1. Establish a connection
try:
    connection = psycopg.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT
    )
    print("Connection to PostgreSQL DB successful")

    # 2. Create a cursor to execute queries
    cursor = connection.cursor()

    # 3. Execute a simple test query
    cursor.execute("SELECT version();")

    # 4. Fetch the result
    db_version = cursor.fetchone()
    print(f"PostgreSQL version: {db_version}")

except OperationalError as e:
    print(f"The error '{e}' occurred")

finally:
    # 5. Clean up: close cursor and connection
    if connection:
        cursor.close()
        connection.close()
        print("PostgreSQL connection is closed")

: 

### **3. The Connection & Cursor Objects**

### Key Objects

- **`connection`**: Manages the connection to the database. You commit or rollback transactions here.

- **`cursor`**: The workhorse. You use it to execute SQL commands and fetch results. Think of it as a pointer to a set of results.

### The Execution Flow

1.  `connection = psycopg.connect(...)`

2.  `cursor = connection.cursor()`

3.  `cursor.execute("SELECT ...")`

4.  `results = cursor.fetchall()` / `cursor.fetchone()` / `cursor.fetchmany(size)`

5.  `connection.commit()` (For INSERT/UPDATE/DELETE)

6.  `cursor.close()`

7.  `connection.close()`

## 4. Implementing CRUD Operations

Let's apply our Week 9 SQL knowledge within Python.

### ⚠️ **CRITICAL: Preventing SQL Injection**

**NEVER** use string formatting (`f""` or `%s`) to build SQL queries with user input!

```python
# 🚨 CATASTROPHICALLY DANGEROUS
user_input = "'; DROP TABLE users; --"
cursor.execute(f"SELECT * FROM users WHERE username = '{user_input}'")
# This executes: SELECT * FROM users WHERE username = ''; DROP TABLE users; --'

# ✅ PERFECTLY SAFE - Use parameterized queries
cursor.execute("SELECT * FROM users WHERE username = %s", (user_input,))
```

**psycopg uses `%s` as a placeholder, regardless of the data type.**

### 4.1. CREATE (INSERT)


In [ ]:
def create_task(title, description, user_id):
    """Inserts a new task into the tasks table."""
    try:
        connection = psycopg.connect(
            dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD, host=DB_HOST, port=DB_PORT
        )
        cursor = connection.cursor()

        # ✅ SAFE: Parameterized query
        insert_query = f"""
        INSERT INTO tasks (title, description, user_id)
        VALUES (%s, %s, %s) RETURNING task_id, created_at;
        """
        # The RETURNING clause is useful for getting auto-generated values (like task_id)
        cursor.execute(insert_query, (title, description, user_id))

        # Fetch the returned data (e.g., the new task_id)
        new_task_data = cursor.fetchone()
        print(f"New task created with ID: {new_task_data[0]} at {new_task_data[1]}")

        # Commit the transaction
        connection.commit()
        print("Transaction committed successfully")

    except (Exception, psycopg.Error) as error:
        print(f"Error while inserting task: {error}")
        if connection:
            connection.rollback() # Undo the changes if anything fails
        
    finally:
        if connection:
            cursor.close()
            connection.close()

# Example usage
create_task("Learn psycopg", "Connect Python to our database", 1)

### 4.2. READ (SELECT)

In [ ]:
def get_user_tasks(user_id):
    """Fetches all tasks for a given user."""
    try:
        connection = psycopg.connect(
            dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD, host=DB_HOST, port=DB_PORT
        )
        cursor = connection.cursor()

        select_query = """
        SELECT task_id, title, description, completed, created_at
        FROM tasks 
        WHERE user_id = %s
        ORDER BY created_at DESC;
        """
        cursor.execute(select_query, (user_id,))

        # Fetch all rows
        user_tasks = cursor.fetchall()
        print(type(user_tasks))
        print(f"Found {len(user_tasks)} tasks for user {user_id}:")
        for task in user_tasks:
            print(f"  - {task[1]} (ID: {task[0]}) - Completed: {task[3]}")
        
        return user_tasks # Return the list of tasks for use in the app

    except (Exception, psycopg.Error) as error:
        print(f"Error fetching tasks: {error}")
        return []
        
    finally:
        if connection:
            cursor.close()
            connection.close()

# Example usage
tasks = get_user_tasks(1)

### 4.3. UPDATE

In [ ]:
def mark_task_complete(task_id):
    """Marks a specific task as completed."""
    try:
        connection = psycopg.connect(
            dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD, host=DB_HOST, port=DB_PORT
        )
        cursor = connection.cursor()

        update_query = """
        UPDATE tasks 
        SET completed = TRUE 
        WHERE task_id = %s;
        """
        cursor.execute(update_query, (task_id,))

        # Check how many rows were affected
        rows_affected = cursor.rowcount
        
        if rows_affected > 0:
            print(f"Successfully marked task {task_id} as complete.")
            connection.commit()
        else:
            print(f"No task found with ID {task_id}. No update performed.")
            # No commit needed if nothing changed

    except (Exception, psycopg.Error) as error:
        print(f"Error updating task: {error}")
        if connection:
            connection.rollback()
        
    finally:
        if connection:
            cursor.close()
            connection.close()

# Example usage
mark_task_complete(5)

### 4.4. DELETE

In [ ]:
def delete_task(task_id):
    """Deletes a specific task."""
    try:
        connection = psycopg.connect(
            dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD, host=DB_HOST, port=DB_PORT
        )
        cursor = connection.cursor()

        delete_query = "DELETE FROM tasks WHERE task_id = %s;"
        cursor.execute(delete_query, (task_id,))

        rows_affected = cursor.rowcount
        
        if rows_affected > 0:
            print(f"Successfully deleted task {task_id}.")
            connection.commit()
        else:
            print(f"No task found with ID {task_id}. Nothing deleted.")

    except (Exception, psycopg.Error) as error:
        print(f"Error deleting task: {error}")
        if connection:
            connection.rollback()
        
    finally:
        if connection:
            cursor.close()
            connection.close()

# Example usage - USE WITH CAUTION!
delete_task(10)

## 5. Using Context Managers for Cleaner Code

The `with` statement automatically handles closing the connection and cursor, even if an error occurs.

In [ ]:
def get_task_count():
    """Demonstrates using the connection as a context manager."""
    try:
        with psycopg.connect(
            dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD, host=DB_HOST, port=DB_PORT
        ) as connection:
            # Create a cursor that itself is a context manager
            with connection.cursor() as cursor:
                # cursor.execute("SELECT * FROM tasks;")
                cursor.execute("SELECT COUNT(*) FROM tasks;")
                count = cursor.fetchone()[0]
                print(f"Total tasks in database: {count}")
                # No need to manually close cursor or connection!
    except OperationalError as e:
        print(f"Connection error: {e}")

get_task_count()

## 6. Best Practices Summary

1.  **✅ Use Parameterized Queries:** Always use `%s` placeholders to prevent SQL injection.

2.  **✅ Manage Connections:** Always close your cursor and connection. Use context managers (`with` blocks) when possible.

3.  **✅ Handle Transactions:** Explicitly `commit()` after INSERT/UPDATE/DELETE. Use `rollback()` on errors.

4.  **✅ Check `rowcount`:** For UPDATE/DELETE, check how many rows were affected to confirm the operation.

5.  **✅ Store Credentials Securely:** Never hardcode credentials in your source code. Use environment variables.

```python
# BETTER: Using environment variables
import os
DB_PASSWORD = os.environ.get('DB_PASSWORD')
```